# Phase 4: behavior on Qwen2.5-7B-Instruct (free Colab T4, 8-bit)

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Clone access works the same as the pilot (`GH_TOKEN` secret, or a public repo).

- Scores all 482 questions × 13 conditions at the first turn, keeps the ones the 7B gets right everywhere,
  then scores every kept question × 13 conditions × 3 pushback templates.
- Progress is checkpointed to Drive every 512 prompts. **If Colab disconnects, just `Run all` again**: it resumes
  where it stopped instead of starting over.
- Outputs land in `MyDrive/vnpronouns/outputs/Qwen2.5-7B-Instruct-8bit/`. The last cell prints a summary to paste back to Claude.
- On a bigger GPU (Colab Pro A100/L4), change `CFG` below to `configs/colab.yaml` (bf16), but then use that same config for every later 7B phase.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
url = f"https://{token}@github.com/GTRAN7/vnpronouns.git" if token else "https://github.com/GTRAN7/vnpronouns.git"
if not os.path.exists('/content/vnpronouns'):
    subprocess.run(['git', 'clone', '-q', url, '/content/vnpronouns'], check=True)
else:
    subprocess.run(['git', '-C', '/content/vnpronouns', 'pull', '-q'], check=True)
%cd /content/vnpronouns
!git log --oneline -1
!pip install -q -e ".[eightbit]"
!nvidia-smi --query-gpu=name,memory.total --format=csv
CFG = "configs/colab_t4.yaml"

## 1. Baseline filter (all questions) + flip rates on every kept question
The long step. Re-run after a disconnect to resume.

In [ ]:
!python -m vnpronouns.run_behavior --config {CFG} 2>&1 | tee -a /content/log_behavior.txt

## 2. Statistics: flip rates with 95% bootstrap CIs, McNemar (Bonferroni ×3), per-pair / gender / template breakdowns
CPU only, takes seconds.

In [ ]:
!python -m vnpronouns.stats --config {CFG} 2>&1 | tee /content/log_stats.txt

## 3. Reality check: free-text replies on 50 questions × 13 conditions (flat template)

In [ ]:
!python -m vnpronouns.freetext --config {CFG} --n-questions 50 2>&1 | tee /content/log_freetext.txt

## 4. Summary to paste back to Claude

In [ ]:
import re
def clean(path, last_run_only=False):
    lines = open(path, encoding='utf-8').read().splitlines()
    if last_run_only:  # the behavior log is appended across resumes; show the final run
        starts = [i for i, l in enumerate(lines) if l.startswith('baseline:')]
        lines = lines[starts[-1]:] if starts else lines
    keep = [l.rstrip() for l in lines
            if not re.search(r'it/s\]|s/it\]|Warning|warn\(|^\s*$|Loading|Fetching|Downloading|%\||^  chunk ', l)]
    return '\n'.join(keep)
for p, last in [('/content/log_behavior.txt', True), ('/content/log_stats.txt', False), ('/content/log_freetext.txt', False)]:
    print(f'===== {p.split("/")[-1]} =====')
    print(clean(p, last))
!ls -la /content/drive/MyDrive/vnpronouns/outputs/Qwen2.5-7B-Instruct-8bit/